# Lab 4: Ứng dụng S3/MinIO trong Data Pipeline

## 🎯 Mục tiêu
- Cấu hình **s3fs** để đọc/ghi MinIO như filesystem
- Hiểu cách Spark / Iceberg / Kafka Connect dùng S3 endpoint
- Tạo bucket và path chuẩn cho data lake (staging, warehouse)

## 📋 Prerequisites
- MinIO đang chạy: `docker compose up -d`
- Đã cài: `boto3`, `s3fs`, `pandas` (trong requirements.txt)


## 1. S3 như filesystem với s3fs

**s3fs** là Python filesystem cho S3 (FUSE-style API), dùng với `fsspec`. Công cụ như PyArrow, Pandas, PyIceberg có thể đọc/ghi S3 qua s3fs.

In [ ]:
import s3fs

fs = s3fs.S3FileSystem(
    client_kwargs={
        "endpoint_url": "http://localhost:9000",
        "aws_access_key_id": "minioadmin",
        "aws_secret_access_key": "minioadmin",
        "region_name": "us-east-1",
    },
    config_kwargs={"signature_version": "s3v4"},
)

print("✅ s3fs đã cấu hình (MinIO).")
print("Buckets:", fs.ls("/"))

## 2. Đọc/ghi file qua s3fs

In [ ]:
bucket = "lab-bucket"
path = f"{bucket}/staging/events.json"

# Ghi file
with fs.open(path, "wb") as f:
    f.write(b'{"event": "test", "source": "s3_lab"}')
print(f"✅ Đã ghi: {path}")

# Đọc lại
with fs.open(path, "rb") as f:
    content = f.read().decode("utf-8")
print("Nội dung:", content)

## 3. Cấu hình Spark với MinIO (tham khảo)

Khi chạy Spark (PySpark hoặc Spark Submit) với S3/MinIO, cần set:

```python
# PySpark example
spark.conf.set("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
spark.conf.set("spark.hadoop.fs.s3a.access.key", "minioadmin")
spark.conf.set("spark.hadoop.fs.s3a.secret.key", "minioadmin")
spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
spark.conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
```

Đường dẫn đọc/ghi: `s3a://bucket-name/path/to/data`

## 4. Cấu hình PyIceberg với MinIO (tham khảo)

Trong PyIceberg Lab, catalog có thể dùng warehouse trên S3:

```python
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "default",
    **{
        "type": "sql",
        "uri": "sqlite:///warehouse/catalog.db",
        "warehouse": "s3://warehouse/",  # MinIO bucket
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
    },
)
```

Cần cài: `pyiceberg[s3fs]` và tạo bucket `warehouse` trong MinIO.

## 5. Cấu trúc bucket cho Data Lake (best practice)

Một cách tổ chức thường gặp:

```
s3://my-datalake/
├── raw/           # Dữ liệu thô (Kafka sink, batch ingest)
├── staging/       # Staging cho ETL
├── warehouse/     # Iceberg/Spark tables
│   └── db/
│       └── table/
└── checkpoint/   # Spark/Flink checkpoint (nếu dùng)
```

Trong MinIO: tạo bucket (vd `my-datalake`) và các prefix tương ứng.

In [ ]:
# Tạo cấu trúc prefix trong bucket (chỉ cần upload 1 object để "có" prefix)
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
    config=Config(signature_version="s3v4"),
)

datalake_bucket = "my-datalake"
try:
    s3.create_bucket(Bucket=datalake_bucket)
    print(f"✅ Bucket '{datalake_bucket}' đã tạo.")
except Exception as e:
    if "AlreadyOwnedByYou" in str(e) or "AlreadyExists" in str(e):
        print(f"Bucket '{datalake_bucket}' đã tồn tại.")
    else:
        print(e)

# Placeholder objects cho cấu trúc thư mục (optional)
for prefix in ["raw/", "staging/", "warehouse/", "checkpoint/"]:
    s3.put_object(Bucket=datalake_bucket, Key=f"{prefix}.keep", Body=b"")
print("Đã tạo prefix: raw/, staging/, warehouse/, checkpoint/")

## 6. Tóm tắt

- **s3fs**: đọc/ghi S3 như file trong Python (pandas, pyarrow tương thích).
- **Spark**: dùng `fs.s3a.*` config + endpoint, path-style.
- **PyIceberg**: warehouse = `s3://bucket/`, thêm `s3.endpoint` và credentials.
- Tổ chức bucket: raw / staging / warehouse / checkpoint.

**Kết thúc S3 Lab.** Có thể kết hợp với PyIceberg Lab hoặc Streaming Lakehouse Lab, cấu hình warehouse trỏ tới MinIO.